In [12]:
import pandas as pd
import numpy as np
import commands
from data_main import Data
from shapely.geometry import Point, LineString
import math
import helper

np.random.seed(42)

In [13]:
args = commands.get_commands()[2]
data = Data(args=args)
data.read_data()

## Demand mapping

In [15]:
def demand_deterministic_landfall(data, track_err, hurr_cat, t):
    """Compute demand for the given Hurricane position, category,
    and the time period in deterministic landfall case.

    hurr_pos : tuple
        (long, lat) as the position of hurricane track.
    hurr_cat: int
        Hurricane category = 0, 1, 2, 3, 4, or 5.
    t : int
        time period = 0, 1, ...,9, or 10.
    """
    DF = np.zeros([data.I])
    # First demand is realized 4 periods before landfall.
    if t >= data.T - data.tau:
        dY = [0] * data.I
        dX = [0] * data.I
        for i in range(data.I):
            hurr_pos =  [data.FORE.X[t] + data.X_rotate * track_err/data.Xmiles,
                         data.FORE.Y[t] + data.Y_rotate * track_err/data.Ymiles]
            proj_point = data.study_line.interpolate(
                data.study_line.project(Point(hurr_pos))
                )
            dY[i] = math.sqrt(
                ((proj_point.x - data.DP_GIS[i, 0]) * data.Xmiles)**2 +
                ((proj_point.y - data.DP_GIS[i, 1]) * data.Ymiles)**2
                )
            dX[i] = math.sqrt(
                ((proj_point.x - hurr_pos[0]) * data.Xmiles)**2 +
                ((proj_point.y - hurr_pos[1]) * data.Ymiles)**2
                )
            if data.demand_zone.contains(Point(hurr_pos)):
                DF_by_int = helper.demandByWS(hurr_cat)
                DF_by_time = 1 - dX[i] / data.x_max
                DF_by_track = 1 - dY[i] / data.study_region_dist
                DF[i] = DF_by_track * DF_by_time * DF_by_int
    return DF.round(3).tolist()

In [16]:
t = 8
s = 1
track_err_oos, intensity_oos = data.test_samples_from_tree[s][t]
demand_deterministic_landfall(data, track_err_oos, intensity_oos, t)

[0.234, 0.261, 0.214, 0.211, 0.21, 0.199, 0.182]

In [17]:
track_err_oos, intensity_oos

(-41.1, 2)

In [18]:
import pandas as pd

def save_real_sample_paths_as_tuples(data, output_file="real_sample_paths.csv"):
    """
    Saves real hurricane sample paths from data.test_samples_from_tree to a CSV file.
    Each state is written as a tuple (track_err, intensity) in one cell per time step.
    Each row is a complete sample path.
    """
    nbOS = len(data.test_samples_from_tree)       # number of sample paths
    T = len(data.test_samples_from_tree[0])       # number of time steps

    rows = []
    for s in range(nbOS):
        row = []
        for t in range(T):
            track_err, intensity = data.test_samples_from_tree[s][t]
            row.append(f"({track_err}, {intensity})")
        rows.append(row)

    # Column headers: t0, t1, ..., t(T-1)
    columns = [f"t{t}" for t in range(T)]

    df = pd.DataFrame(rows, columns=columns)
    df.to_csv(output_file, index=False)
    print(f"Saved {nbOS} sample paths with tuple-format states to '{output_file}'.")


In [19]:
save_real_sample_paths_as_tuples(data)


Saved 1000 sample paths with tuple-format states to 'real_sample_paths.csv'.


In [11]:
def in_sample(t, state, S=100):
    dim = len(state)
    in_sample = np.zeros([S, data.T, dim])
    for s in range(S):
        sampled_state = state
        in_sample[s, t, :] = state
        for t_ in range(t + 1, data.T):
            a = range(len(data.state_space[t_]))
            p = list(data.pi_mssp[t_ - 1][sampled_state].values())
            sampled_index = np.random.choice(a=a, p=p,)
            sampled_state = data.state_space[t_][sampled_index]
            in_sample[s, t_, :] = sampled_state
    return in_sample

in_sample_data = in_sample(5, data.state_space[5][0], S=100)


In [22]:
initial_state = data.state_space[0][0]
in_sample_data = in_sample(0, initial_state, S=100)

all_demands = []

for s in range(100):
    sample_demands = []
    for t in range(data.T):
        track_err, intensity = in_sample_data[s, t]
        demand = demand_deterministic_landfall(data, track_err, intensity, t)
        sample_demands.append(demand)
    all_demands.append(sample_demands)

# Population per location
population = np.array([66793,411809,265482,429384,173616,39168,35102])

# Convert all_demands to NumPy array
all_demands_np = np.array(all_demands)  # shape: (5, T, 7)

# Multiply by population (broadcast across last axis)
scaled_demands = all_demands_np * population  # shape: (5, T, 7)

for s in range(5):
    print(f"\nScenario {s}:")
    for t in range(data.T):
        vector = np.round(scaled_demands[s, t], 3).tolist()
        print(f"  Time {t}: {vector}")


# for s in range(5):
#     print(f"\nScenario {s}:")
#     for t in range(data.T):
#         demand_vector = all_demands[s][t]
#         print(f"  Time {t}: {demand_vector}")




Scenario 0:
  Time 0: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Time 1: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Time 2: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Time 3: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Time 4: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Time 5: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Time 6: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Time 7: [3873.994, 28414.821, 13539.582, 21469.2, 8680.8, 1801.728, 1404.08]
  Time 8: [10620.087, 77008.283, 36901.998, 58396.224, 23438.16, 4856.832, 3755.914]
  Time 9: [22108.483, 156899.229, 78582.672, 124521.36, 50175.024, 10536.192, 8389.378]
  Time 10: [15763.148, 113247.475, 55485.738, 87594.336, 35244.048, 7363.584, 5791.83]

Scenario 1:
  Time 0: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Time 1: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Time 2: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Time 3: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Time 4: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Time 5: [2538.134, 18531.405, 8760.906, 13740.288, 5555.712, 1175.04, 912.652]
  

In [23]:
import pandas as pd
import os
import numpy as np

# Create output directory
os.makedirs("csv_by_location", exist_ok=True)

# scaled_demands shape: (5 scenarios, T time steps, 7 locations)
S, T, I = scaled_demands.shape

# Loop over each location
for i in range(I):
    data_matrix = []  # collect demand per scenario for location i
    for s in range(S):
        row = scaled_demands[s, :, i]  # shape: (T,)
        data_matrix.append(np.round(row, 3))  # rounded to 3 decimals

    df = pd.DataFrame(data_matrix)
    df.to_csv(f"csv_by_location/location_{i}.csv", index=False, header=False)


In [ ]:
# in_sample_data[10][8]

array([-178.03,    1.  ])

In [ ]:
# data.state_space[5][0]

(-145.97, 0)